# 01 — Setup

Creates the Delta tables that the rest of the project writes to. Run **once** after the observability lakehouse exists and `config.json` has been uploaded to `Files/config.json` of the observability lakehouse (or to the notebook resource folder).

In [ ]:
import json, os
from pyspark.sql import functions as F

# When run inside Fabric, the notebook resource folder contains config.json.
# Fabric exposes notebook-attached files via mssparkutils / notebookutils.
try:
    import notebookutils  # type: ignore
    cfg_path = notebookutils.nbResPath + '/builtin/config.json'
    if not os.path.exists(cfg_path):
        # Fallback: lakehouse Files/config.json
        cfg_path = '/lakehouse/default/Files/config.json'
except Exception:
    cfg_path = './config.json'

with open(cfg_path, 'r', encoding='utf-8') as f:
    CFG = json.load(f)

OBS_WS  = CFG['observability_workspace_name']
OBS_LH  = CFG['observability_lakehouse_name']
TBL     = CFG['tables']
API     = CFG['fabric_api']
# monitored_workspaces is a list of workspace display names (strings).
# Backwards-compat: also accept the old [{workspace_name: ...}] shape.
_raw_mon = CFG['monitored_workspaces']
MONITOR = [m if isinstance(m, str) else m['workspace_name'] for m in _raw_mon]
INGEST  = CFG['ingestion']
print(f'Observability workspace : {OBS_WS}')
print(f'Observability lakehouse : {OBS_LH}')
print(f'Monitored workspaces    : {MONITOR}')

## Create empty Delta tables (idempotent)

In [ ]:
from pyspark.sql.types import *

bronze_schema = StructType([
    StructField('event_date', DateType()),
    StructField('source_workspace_name', StringType()),
    StructField('workspaceId', StringType()),
    StructField('itemId', StringType()),
    StructField('itemType', StringType()),
    StructField('tenantId', StringType()),
    StructField('executingPrincipalId', StringType()),
    StructField('executingUPN', StringType()),
    StructField('executingPrincipalType', StringType()),
    StructField('correlationId', StringType()),
    StructField('operationName', StringType()),
    StructField('operationCategory', StringType()),
    StructField('accessStartTime', TimestampType()),
    StructField('accessEndTime', TimestampType()),
    StructField('originatingApp', StringType()),
    StructField('serviceEndpoint', StringType()),
    StructField('Resource', StringType()),
    StructField('capacityId', StringType()),
    StructField('httpStatusCode', IntegerType()),
    StructField('isShortcut', BooleanType()),
    StructField('accessedViaResource', StringType()),
    StructField('callerIPAddress', StringType()),
    StructField('contentLength', LongType()),
    StructField('_source_file', StringType()),
    StructField('_ingested_at', TimestampType()),
])

shortcut_schema = StructType([
    StructField('scanned_at', TimestampType()),
    StructField('source_workspace_name', StringType()),
    StructField('source_workspace_id', StringType()),
    StructField('source_item_name', StringType()),
    StructField('source_item_id', StringType()),
    StructField('shortcut_name', StringType()),
    StructField('shortcut_path', StringType()),
    StructField('shortcut_full_path', StringType()),
    StructField('target_type', StringType()),
    StructField('target_workspace_id', StringType()),
    StructField('target_item_id', StringType()),
    StructField('target_path', StringType()),
    StructField('target_connection_id', StringType()),
    StructField('target_location', StringType()),
    StructField('target_subpath', StringType()),
    StructField('target_raw_json', StringType()),
])

for name, schema in [(TBL['bronze'], bronze_schema), (TBL['dim_shortcuts'], shortcut_schema)]:
    if not spark.catalog.tableExists(name):
        (spark.createDataFrame([], schema)
             .write.format('delta').mode('overwrite').saveAsTable(name))
        print(f'Created table: {name}')
    else:
        print(f'Exists       : {name}')

# silver is created by 04_silver_enriched_access on first run (CTAS)
print('Setup complete.')